# Real Data Experiments of NeurIPS 2023 Paper

Reproduces Table 2 and Figure 3 of the NeurIPS 2023 paper comparing:

- Maximisisation of the marginal $(\sigma^2, \tau^2)$ posterior via EM algorithm with half-Cauchy prior on $\tau$ (`RidgeEM(t2=False)`)
- LOOCV Ridge regression

and the `NEURIPS2023` named problem collections.

All experiments take substantial time and are therefore not run during integration testing.

## d = 1

In [ ]:
import numpy as np
from fastridge import RidgeEM, RidgeLOOCV
from experiments import EmpiricalDataExperiment
from problems import NEURIPS2023
from data import DATASETS

# estimator indices: 0=EM, 1=CV_fix, 2=CV_glm
estimators = [
    RidgeEM(t2=False),
    RidgeLOOCV(alphas=np.logspace(-10, 10, 100, base=10)),
    RidgeLOOCV(alphas=100),
]
est_names = ['EM', 'CV_fix', 'CV_glm']

problems_d1 = sorted(NEURIPS2023, key=lambda p: DATASETS[p.dataset]['n'])
exp_d1 = EmpiricalDataExperiment(
    problems_d1, estimators, n_iterations=100, seed=123,
    est_names=est_names, verbose=True).run()
print()

In [ ]:
import pandas as pd

rows_d1 = []
for i, problem in enumerate(exp_d1.problems):
    em_time  = np.nanmean(exp_d1.fitting_time_[:, i, 0, 0])
    cv_time  = (np.nanmean(exp_d1.fitting_time_[:, i, 0, 1])
               + np.nanmean(exp_d1.fitting_time_[:, i, 0, 2])) / 2
    row = {'dataset': problem.dataset, 'target': problem.target, 'd': 1}
    for j, name in enumerate(exp_d1.est_names):
        row[name] = np.nanmean(exp_d1.prediction_r2_[:, i, 0, j])
    row['Speed Up Ratio'] = cv_time / em_time
    row['p']              = np.nanmean(exp_d1.number_of_features_[:, i, 0, 0])
    row['n_train']        = int(exp_d1.ns[i, 0])
    row['n:p']            = int(exp_d1.ns[i, 0]) / np.nanmean(exp_d1.number_of_features_[:, i, 0, 0])
    rows_d1.append(row)
df_d1 = pd.DataFrame(rows_d1).sort_values('n_train', ascending=False).round(2)
df_d1

## d = 2

In [ ]:
from problems import NEURIPS2023_D2

problems_d2 = sorted(NEURIPS2023_D2, key=lambda p: DATASETS[p.dataset]['n'])
exp_d2 = EmpiricalDataExperiment(
    problems_d2, estimators, n_iterations=100, seed=123, polynomial=2,
    est_names=est_names, verbose=True).run()
print()

In [ ]:
rows_d2 = []
for i, problem in enumerate(exp_d2.problems):
    em_time  = np.nanmean(exp_d2.fitting_time_[:, i, 0, 0])
    cv_time  = (np.nanmean(exp_d2.fitting_time_[:, i, 0, 1])
               + np.nanmean(exp_d2.fitting_time_[:, i, 0, 2])) / 2
    row = {'dataset': problem.dataset, 'target': problem.target, 'd': 2}
    for j, name in enumerate(exp_d2.est_names):
        row[name] = np.nanmean(exp_d2.prediction_r2_[:, i, 0, j])
    row['Speed Up Ratio'] = cv_time / em_time
    row['p']              = np.nanmean(exp_d2.number_of_features_[:, i, 0, 0])
    row['n_train']        = int(exp_d2.ns[i, 0])
    row['n:p']            = int(exp_d2.ns[i, 0]) / np.nanmean(exp_d2.number_of_features_[:, i, 0, 0])
    rows_d2.append(row)
df_d2 = pd.DataFrame(rows_d2).sort_values('n_train', ascending=False).round(2)
df_d2

## d = 3

In [ ]:
from problems import NEURIPS2023_D3

problems_d3 = sorted(NEURIPS2023_D3, key=lambda p: DATASETS[p.dataset]['n'])
exp_d3 = EmpiricalDataExperiment(
    problems_d3, estimators, n_iterations=100, seed=123, polynomial=3,
    est_names=est_names, verbose=True).run()
print()

In [ ]:
rows_d3 = []
for i, problem in enumerate(exp_d3.problems):
    em_time  = np.nanmean(exp_d3.fitting_time_[:, i, 0, 0])
    cv_time  = (np.nanmean(exp_d3.fitting_time_[:, i, 0, 1])
               + np.nanmean(exp_d3.fitting_time_[:, i, 0, 2])) / 2
    row = {'dataset': problem.dataset, 'target': problem.target, 'd': 3}
    for j, name in enumerate(exp_d3.est_names):
        row[name] = np.nanmean(exp_d3.prediction_r2_[:, i, 0, j])
    row['Speed Up Ratio'] = cv_time / em_time
    row['p']              = np.nanmean(exp_d3.number_of_features_[:, i, 0, 0])
    row['n_train']        = int(exp_d3.ns[i, 0])
    row['n:p']            = int(exp_d3.ns[i, 0]) / np.nanmean(exp_d3.number_of_features_[:, i, 0, 0])
    rows_d3.append(row)
df_d3 = pd.DataFrame(rows_d3).sort_values('n_train', ascending=False).round(2)
df_d3

## Summary

In [ ]:
df_all = pd.concat([df_d1, df_d2, df_d3]).sort_values('n_train', ascending=False).round(2)
df_all

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

def make_figure3(exp_d1, exp_d2, exp_d3, output_path=None):
    """2x3 scatter of EM vs CV R² for d=1,2,3. Color = speed-up ratio.

    Top row: CV_glm on y-axis. Bottom row: CV_fix on y-axis.
    Values below CLIP_MIN=-0.1 are clipped and shown with a dashed edge.
    Estimator indices: 0=EM, 1=CV_fix, 2=CV_glm.
    """
    experiments = [exp_d1, exp_d2, exp_d3]
    # (cv_est_idx, cv_label)
    cv_rows = [(2, "CV GLM Grid"), (1, "CV Fixed Grid")]
    CLIP_MIN = -0.1
    PAD      =  0.03

    all_su = [
        np.nanmean(exp.fitting_time_[:, i, 0, cv]) / np.nanmean(exp.fitting_time_[:, i, 0, 0])
        for exp in experiments
        for cv, _ in cv_rows
        for i in range(len(exp.problems))
    ]
    norm = mcolors.Normalize(vmin=min(all_su), vmax=max(all_su))
    cmap = plt.cm.Greens

    fig, axes = plt.subplots(2, 3, figsize=(9, 5.4), sharex=True, sharey=True)
    fig.subplots_adjust(left=0.11, right=0.84, bottom=0.11, top=0.93,
                        hspace=0.06, wspace=0.04)

    for col, exp in enumerate(experiments):
        n_probs = len(exp.problems)
        for row, (cv_idx, cv_label) in enumerate(cv_rows):
            ax = axes[row, col]

            true_em = [np.nanmean(exp.prediction_r2_[:, i, 0, 0]) for i in range(n_probs)]
            true_cv = [np.nanmean(exp.prediction_r2_[:, i, 0, cv_idx]) for i in range(n_probs)]
            su      = [np.nanmean(exp.fitting_time_[:, i, 0, cv_idx])
                       / np.nanmean(exp.fitting_time_[:, i, 0, 0]) for i in range(n_probs)]
            disp_em = [max(CLIP_MIN, x) for x in true_em]
            disp_cv = [max(CLIP_MIN, y) for y in true_cv]
            clipped = [e < CLIP_MIN or c < CLIP_MIN
                       for e, c in zip(true_em, true_cv)]
            colors  = cmap(norm(np.array(su)))

            idx_in  = [i for i, cl in enumerate(clipped) if not cl]
            idx_out = [i for i, cl in enumerate(clipped) if cl]

            if idx_in:
                ax.scatter([disp_em[i] for i in idx_in],
                           [disp_cv[i] for i in idx_in],
                           c=[colors[i] for i in idx_in],
                           s=50, zorder=3, edgecolors="k", linewidths=0.6)
            if idx_out:
                sc = ax.scatter([disp_em[i] for i in idx_out],
                                [disp_cv[i] for i in idx_out],
                                c=[colors[i] for i in idx_out],
                                s=50, zorder=4, edgecolors="k", linewidths=0.8)
                sc.set_linestyle("--")

            ax.plot([CLIP_MIN, 1], [CLIP_MIN, 1], "k--", lw=0.8, zorder=2)
            ax.axhline(0, color="0.8", lw=0.5, zorder=1)
            ax.axvline(0, color="0.8", lw=0.5, zorder=1)
            ax.set_xlim(CLIP_MIN - PAD, 1 + PAD)
            ax.set_ylim(CLIP_MIN - PAD, 1 + PAD)

            if row == 1:
                ax.set_xlabel('BayesEM $R^2$')
                ax.set_xticks([0.0, 0.5, 1.0])
            if col == 0:
                ax.set_ylabel(f'{cv_label} $R^2$')
                ax.set_yticks([0.0, 0.5, 1.0])
            if row == 0:
                ax.set_title(f'$d = {col + 1}$')

    cbar_ax = fig.add_axes([0.86, 0.28, 0.02, 0.46])
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    cbar = fig.colorbar(sm, cax=cbar_ax)
    cbar.ax.yaxis.set_label_position("right")
    cbar.set_label('speed-up ratio', rotation=90, labelpad=-30)

    if output_path:
        fig.savefig(output_path, bbox_inches='tight')

make_figure3(exp_d1, exp_d2, exp_d3,
             output_path='../output/paper2023_figure3.pdf')

In [ ]:
df_all.to_csv('../output/paper2023_table2.csv', index=False)